# 공공데이터 포털 API 호출 통합 테스트 (신규 구조)

이 노트북은 새로 작성된 `api_client.py`를 사용하여 공공데이터 포털의 다양한 API를 구조화된 방식으로 호출하는 테스트를 수행합니다.

### 업데이트 내용
- `rows` 파라미터를 사용하여 가져올 데이터 건수를 직접 지정할 수 있습니다.

In [0]:
import json
from api_client import ATAPIClient, NCSWholesaleClient
import pandas as pd

import time
import os
from datetime import datetime, timedelta

# 1. API 키 설정
ncs_key = 'af8b2ec5a3e9bfaf66ec313ed2e3ad9813ea28eecaa6318ca292050b5e1caf60'
at_key = '14392c909baea8f8bf2455b1e379788cdfe28f4ef535a4d8e87e576a81d9ffbc'

# 2. 클라이언트 초기화
at_client = ATAPIClient(at_key)
ncs_client = NCSWholesaleClient(ncs_key)

### 2. aT 일자별 가격 정보
- 마찬가지로 `rows`를 사용하여 결과 건수를 조절합니다.

In [0]:
# 일자별 가격 구체화
import time
import pandas as pd

print("--- 5. aT 일자별 상세 정보 호출 시작 ---")

# 품목코드 참조 파일 불러오기 (앞서 추출한 csv 활용)
ref_file_path = '/Workspace/Users/biod1614@gmail.com/asac_10_dataanalysis/API/ref_sheet_품목코드.csv'
try:
    df_ref = pd.read_csv(ref_file_path, encoding='utf-8-sig') 
except:
    df_ref = pd.read_csv(ref_file_path, encoding='cp949')

date_gte = "20240101" # 검색 시작일 (원하는 날짜로 변경하세요)
date_lte = "20241231" # 검색 종료일 (원하는 날짜로 변경하세요)

all_day_items = []

# 모든 부류, 품목 코드 순회
for index, row in df_ref.iterrows():
    ctgry_cd = str(row['부류코드'])
    item_cd = str(row['품목코드']).zfill(3) # 품목코드는 3자리
    item_nm = row['품목명']
    
    print(f"\n[수집 진행] 부류: {ctgry_cd}, 품목: {item_cd} ({item_nm})")
    
    day_price_params = {
        "date_gte": date_gte,
        "date_lte": date_lte,
        "ctgry_cd": ctgry_cd,
        "item_cd": item_cd,
    }

    page = 1
    rows_per_page = 1000
    item_count = 0
    
    while True:
        day_data = at_client.fetch_day_price(
            **day_price_params,
            page=page,
            rows=rows_per_page,
            return_type="json"
        )
        
        if not day_data:
            break
            
        body = day_data.get('response', {}).get('body', {})
        items = body.get('items', {}).get('item', [])
        total_count = body.get('totalCount', 0)
        
        if isinstance(items, dict):
            items = [items]
            
        if not items:
            break
            
        all_day_items.extend(items)
        item_count += len(items)
        
        if item_count >= int(total_count):
            print(f" -> {item_nm} 데이터 {item_count}/{total_count}개 수집 완료.")
            break
            
        page += 1
        time.sleep(0.05) # API 호출 제한 방지를 위한 짧은 대기

print(f"\n모든 품목 수집 완료. 총 {len(all_day_items)}건 수집됨.")

# 표 그리기 및 CSV 저장
if all_day_items:
    df_day = pd.DataFrame(all_day_items)
    save_path = "/Workspace/Users/biod1614@gmail.com/asac_10_dataanalysis/API/price_change.csv"
    df_day.to_csv(save_path, encoding='cp949', index=False)
    print(f"\n전체 데이터 '{save_path}' 파일로 저장 완료.")
    display(df_day)
else:
    print("조건에 맞는 데이터가 존재하지 않습니다.")


### 3. aT 품목,지역별 가격 정보
- 마찬가지로 `rows`를 사용하여 결과 건수를 조절합니다.

In [0]:
### 4. aT 지역별 세부 가격 정보 호출 테스트 (신규 전용 메서드 사용)
print("--- 4. aT 지역별 세부 정보 호출 시작 ---")

# 새로 작성된 전용 메서드를 통해 직관적으로 파라미터를 설정합니다.
regional_data = at_client.fetch_region_price(
    # [필수 파라미터]
    date_gte="20250101",   # 조회할 시작일 
    date_lte="20260408",   # 조회할 종료일 
    sgg_cd="1101",         # 시군구코드 (예: 1101=서울 등의 지역코드)
    
    # [선택 필터 파라미터] - 필요 없는 항목은 빼거나 None 유지
    # se_cd="01",            # 01: 소매, 02: 중도매(도매)
    # ctgry_cd="100",      # 식량작물 등 부류코드
    # item_cd="111",       # 쌀 등 품목코드
    
    # [응답 제어]
    # selectable="exmn_ymd,item_nm,se_nm,sgg_nm,exmn_dd_avg_prc", # 원하는 컬럼만 추출
    rows=10,                # 반환받을 데이터 건수 (최대 1000)
    return_type="json"
)

if regional_data:
    items = regional_data.get('response', {}).get('body', {}).get('items', {}).get('item', [])
    print(f"성공적으로 {len(items)}건의 데이터를 수신했습니다.\n")
    if items:
        
        import json
        print(json.dumps(items, indent=4, ensure_ascii=False))


### 3. aT 출하량 정보
아직 계획설정 없음

In [0]:
# 출하일 추이 정보 호출 (날짜 범위 순환)

print("--- 6. aT 출하일 추이 정보 호출 시작 (날짜 범위 순환) ---")

# 검색 기간 설정 (원하시는 날짜로 변경하세요)
start_date_str = "20250101"
end_date_str = "20260408"

# 날짜 리스트 생성
start_date = datetime.strptime(start_date_str, "%Y%m%d")
end_date = datetime.strptime(end_date_str, "%Y%m%d")
date_list = [(start_date + timedelta(days=x)).strftime("%Y%m%d") for x in range((end_date - start_date).days + 1)]

output_dir = "/Workspace/Users/biod1614@gmail.com/asac_10_dataanalysis/API"
checkpoint_file = "/Workspace/Users/biod1614@gmail.com/asac_10_dataanalysis/API/Shipment_trend_checkpoint.csv"

# 기존 처리 날짜 읽어오기 (이어하기용)
processed_dates = set()
last_processed_date = None
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r', encoding='utf-8') as f:
        # 파일을 한 번만 읽어서 리스트로 저장 (파일 커서 문제 방지)
        lines = [line.strip() for line in f if line.strip()]
    processed_dates = set(lines)
    if lines:
        last_processed_date = lines[-1]
print(f"기존에 수집 완료된 날짜 수: {len(processed_dates)}개. 마지막 처리 날짜: {last_processed_date}")
print("이어서 시작합니다.\n")

# 체크포인트의 마지막 날짜 이후부터 호출
if last_processed_date and last_processed_date in date_list:
    start_idx = date_list.index(last_processed_date) + 1
    pending_dates = date_list[start_idx:]
else:
    pending_dates = [d for d in date_list if d not in processed_dates]

total_processed_days = 0

for target_date in pending_dates:
    print(f"\n[수집 진행] 출하일자: {target_date}")
    
    page = 1
    rows_per_page = 1000
    day_items = []
    is_success = True
    
    while True:
        # fetch_shipment_trend 사용
        shipment_data = at_client.fetch_shipment_trend(
            spmt_ymd=target_date,
            page=page,
            rows=rows_per_page,
            return_type="json"
        )
        
        if shipment_data is None:
            print(f" -> {target_date} 호출 중 에러 발생. 중단합니다.")
            is_success = False
            break
            
        body = shipment_data.get('response', {}).get('body', {})
        items = body.get('items', {}).get('item', [])
        total_count = body.get('totalCount', 0)
        
        if isinstance(items, dict):
            items = [items]
            
        if not items:
            break
            
        day_items.extend(items)
        
        if len(day_items) >= int(total_count):
            break
            
        page += 1
        time.sleep(0.05)
        
    if not is_success:
        break
        
    # 날짜별 수집 완료 시 월별 파일에 저장 (파일 크기 제한 방지)
    if day_items:
        df_day_items = pd.DataFrame(day_items)
        month_suffix = target_date[:6]  # YYYYMM
        output_csv = os.path.join(output_dir, f"Shipment_trend_{month_suffix}.csv")
        write_header = not os.path.exists(output_csv)
        df_day_items.to_csv(output_csv, mode='a', index=False, encoding='cp949', header=write_header)
        print(f" -> {target_date}: {len(day_items)}건 저장 완료. ({output_csv})")
    else:
        print(f" -> {target_date}: 데이터 없음.")
        
    # 체크포인트 기록
    with open(checkpoint_file, 'a', encoding='utf-8') as f:
        f.write(f"{target_date}\n")
    processed_dates.add(target_date)
    total_processed_days += 1

print(f"\n수집 완료. 이번 실행에서 총 {total_processed_days}일치 데이터를 작업했습니다.")

# 결과 확인 (월별 파일 목록)
import glob
csv_files = sorted(glob.glob(os.path.join(output_dir, "Shipment_trend_*.csv")))
csv_files = [f for f in csv_files if "checkpoint" not in f]
total_rows = 0
for csv_file in csv_files:
    try:
        row_count = sum(1 for _ in open(csv_file, encoding='cp949')) - 1  # 헤더 제외
        total_rows += row_count
        print(f"  {os.path.basename(csv_file)}: {row_count}건")
    except:
        pass
print(f"\n=== 현재까지 총 누적 데이터 수: {total_rows}건 ===")